# Hands-on 7: Membersihkan Timestamp, Resampling, dan Imputasi Sederhana

Buku Referensi: *IoT Terintegrasi dengan Machine Learning & Data Analytics* (Onno W. Purbo), Bab 4.

Alur: parse timestamp -> buang invalid -> urutkan -> hapus duplikat -> resample 10 menit ->
imputasi (forward fill vs interpolasi linier) -> simpan `data/processed/room_sensor_clean.csv`.

## Tujuan

Pada latihan ini, pembaca akan:

- membersihkan kolom waktu,
- menghapus duplikasi waktu,
- menangani urutan waktu yang berantakan,
- melakukan resampling ke interval target,
- dan mengisi data hilang dengan imputasi sederhana.

Ini sangat penting karena data IoT nyata sering datang tidak rapi, dengan interval tidak konsisten dan nilai hilang.

Di sisi teori, An Introduction to IoT Analytics membahas bahwa time series bisa bersifat non-stasioner, memiliki tren dan seasonality, sehingga penanganan waktu yang benar menjadi hal mendasar sebelum model apapun diterapkan.

## 2. Contoh nyata yang bisa diimplementasikan

Ruang server

Sensor seharusnya mengirim tiap 10 menit, tetapi kadang: ● terlambat, ● ganda, ● atau hilang karena koneksi putus.

Greenhouse

Sensor tanah kadang mengirim lebih cepat setelah penyiraman, lalu lambat saat baterai lemah.

Monitoring kendaraan

GPS kadang hilang saat melewati area tertutup, lalu muncul lagi dengan lompatan waktu. Semua kasus ini butuh pembersihan waktu dan resampling agar data siap dianalisis.

In [1]:
import os
from pathlib import Path

while not (Path.cwd() / "scripts" / "mqtt").is_dir():
    os.chdir("..")
print("[INFO] Working directory:", Path.cwd())

[INFO] Working directory: /downloads/iot-ml-book


## 1. Bangkitkan Dataset Kotor (masalah khas IoT)

In [2]:
import subprocess, sys
r = subprocess.run([sys.executable, "scripts/generate_dirty_dataset.py"],
                   capture_output=True, text=True)
print(r.stdout)

[SUKSES] Dataset kotor tersimpan: data/raw/room_sensor_dirty.csv (1003 baris)
[INFO] Masalah disuntikkan - invalid_timestamp: 20
[INFO] Masalah disuntikkan - duplikat: 15
[INFO] Masalah disuntikkan - nilai_hilang: 120
[INFO] Masalah disuntikkan - baris_gap: 20



## 2. Cleaning, Resampling 10 Menit, dan Imputasi

In [3]:
r = subprocess.run([sys.executable, "scripts/clean_timestamp_resample.py"],
                   capture_output=True, text=True)
print(r.stdout)

=== Data awal ===
             timestamp device_id  ...  co2_ppm  power_watt
0  2026-01-03 05:40:00    room01  ...    453.0       87.60
1  2026-01-01 15:50:00    room01  ...    773.0      140.14
2  2026-01-06 20:30:00    room01  ...    430.0       88.67
3          bukan-waktu    room01  ...    887.0      160.93
4  2026-01-05 19:20:00    room01  ...    450.0       87.96

[5 rows x 6 columns]
Jumlah baris awal: 1003

=== Setelah membersihkan timestamp ===
              timestamp device_id  ...  co2_ppm  power_watt
564 2026-01-01 00:00:00    room01  ...    431.0       85.57
785 2026-01-01 00:10:00    room01  ...    457.0       83.39
360 2026-01-01 00:20:00    room01  ...    431.0       90.09
427 2026-01-01 00:30:00    room01  ...    437.0       90.72
435 2026-01-01 00:40:00    room01  ...    453.0       82.10

[5 rows x 6 columns]
Jumlah baris setelah clean timestamp: 968



## 3. Perbandingan Visual Sebelum/Sesudah

In [4]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

df_dirty = pd.read_csv("data/raw/room_sensor_dirty.csv")
df_dirty["timestamp"] = pd.to_datetime(df_dirty["timestamp"], errors="coerce")
df_dirty = df_dirty.dropna(subset=["timestamp"]).sort_values("timestamp")
df_clean = pd.read_csv("data/processed/room_sensor_clean.csv", parse_dates=["timestamp"])

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
axes[0].plot(df_dirty["timestamp"], df_dirty["temperature_c"], ".", ms=2, color="tomato")
axes[0].set_title("Sebelum Cleaning - Suhu (timestamp kotor, duplikat, gap, NaN)")
axes[0].set_ylabel("Suhu (C)")
axes[1].plot(df_clean["timestamp"], df_clean["temperature_c"], "-", lw=1, color="seagreen")
axes[1].set_title("Sesudah Cleaning - Resample 10 menit + Interpolasi Linier")
axes[1].set_ylabel("Suhu (C)")
axes[1].set_xlabel("Waktu")
plt.xticks(rotation=45)
plt.tight_layout()
Path("outputs/figures").mkdir(parents=True, exist_ok=True)
plt.savefig("outputs/figures/cleaning_before_after.png")
plt.show()
print("[SUKSES] outputs/figures/cleaning_before_after.png")

[SUKSES] outputs/figures/cleaning_before_after.png


## Panduan Lengkap dari Buku

_Bagian ini dikutip dari buku PDF (sumber kebenaran praktikum) dan diformat ulang agar mudah dibaca. Kode asli buku tidak diubah; bagian instalasi/persiapan lingkungan diringkas karena sudah dijelaskan di README._

### 3. Dataset kotor untuk latihan

Simpan file ini sebagai:

data/raw/room_sensor_dirty.csv

Isi:

timestamp,device_id,temperature_c,humidity_pct,co2_ppm,p ower_watt 2026-01-01 08:00:00,room01,24.5,58,520,110 2026-01-01 08:10:00,room01,24.7,57,525,112 2026-01-01 08:25:00,room01,25.1,56,540,115 2026-01-01 08:30:00,room01,25.8,55,560,118 2026-01-01 08:30:00,room01,25.9,55,562,119 2026-01-01 08:55:00,room01,26.8,53,620,126 2026-01-01 09:00:00,room01,27.1,52,650,130 2026-01-01 09:20:00,room01,28.1,50,710,139 2026-01-01 09:30:00,room01,,49,740,143 2026-01-01 09:40:00,room01,28.8,,770,147 2026-01-01 09:50:00,room01,29.0,47,,150 2026-01-01 10:00:00,room01,29.2,46,810, bad_timestamp,room01,29.0,47,800,150 2026-01-01 10:20:00,room01,28.7,48,780,146 2026-01-01 10:50:00,room01,27.4,51,690,134

Dataset ini sengaja memiliki:

●   timestamp rusak, ●   duplikasi waktu, ●   interval tidak konsisten, ●   beberapa nilai kosong.

### 4. Source code utama: membersihkan waktu

Buat file:

```python
scripts/clean_timestamp_resample.py
```

Isi:

*(Kode lengkap tersedia pada sel kode notebook ini.)*

Yang dilakukan kode

*(Kode lengkap tersedia pada sel kode notebook ini.)*

### 5. Set index waktu dan cek celah data

Tambahkan:

```python
df = df.set_index("timestamp")
print("\n=== Selisih antar timestamp ===")
print(df.index.to_series().diff())
```

Ini membantu pembaca melihat apakah sensor benar-benar konsisten mengirim data.

### 6. Resampling ke interval 10 menit

Dalam praktik IoT, kita sering ingin mengubah data “berantakan” menjadi interval tetap agar lebih mudah dianalisis. Literatur Perros juga menunjukkan pentingnya pengolahan time series untuk mendukung forecasting, termasuk pengenalan pola musiman dan transformasi data.

Tambahkan:

```python
# Resampling per 10 menit
df_resampled = df.resample("10min").mean(numeric_only=True)
# Untuk kolom kategori seperti device_id, isi dengan forward
fill
df_resampled["device_id"] = "room01"
print("\n=== Data setelah resampling 10 menit ===")
print(df_resampled.head(12))
```

Apa itu resampling?

Resampling berarti memetakan data ke interval waktu baru yang konsisten, misalnya:

- 10 menit,
- 1 menit,
- 1 jam,
- atau 1 hari.

### 7. Imputasi sederhana untuk nilai hilang

Sekarang isi nilai kosong secara bertahap.

### 7.1. Forward fill

df_ffill = df_resampled.copy() df_ffill[["temperature_c", "humidity_pct", "co2_ppm", "power_watt"]] = ( df_ffill[["temperature_c", "humidity_pct", "co2_ppm", "power_watt"]].ffill() )

print("\n=== Setelah forward fill ===") print(df_ffill.head(12))

### 7.2. Interpolasi linier

```python
df_interp = df_resampled.copy()
df_interp[["temperature_c", "humidity_pct", "co2_ppm",
"power_watt"]] = (
    df_interp[["temperature_c", "humidity_pct", "co2_ppm",
"power_watt"]].interpolate(method="linear")
)
print("\n=== Setelah interpolasi linier ===")
print(df_interp.head(12))
```

Kapan pakai apa?

- Forward fill cocok bila nilai tidak berubah drastis dan sensor memang bersifat stabil.
- Interpolasi linier cocok bila perubahan di antara dua titik masuk akal diasumsikan

mulus.

### 8. Simpan hasil data bersih

Tambahkan:

```python
df_interp.to_csv(OUTPUT_PATH)
print(f"\nData bersih disimpan ke: {OUTPUT_PATH}")
```

### 9. Versi lengkap skrip

Berikut versi lengkapnya:

*(Kode lengkap tersedia pada sel kode notebook ini.)*

## Langkah Kerja dari Buku

### 10. Langkah kerja yang harus dilakukan pembaca

1. Baca file kotor.
2. Parse timestamp.
3. Buang waktu invalid.
4. Urutkan data dan hapus duplikasi.
5. Set index waktu.
6. Lakukan resample ke interval target.
7. Lakukan imputasi sederhana.
8. Simpan hasil ke folder processed.

## Penutup dari Buku

Dengan latihan ini, pembaca belajar bahwa data sensor yang baik hampir selalu dibuat, bukan ditemukan dalam keadaan sempurna. Membersihkan waktu, menyamakan interval, dan mengisi nilai hilang adalah pekerjaan inti seorang praktisi IoT analytics. Tanpa ini, grafik bisa menipu, statistik bisa bias, dan model bisa belajar pola yang salah. Itulah mengapa time series handling sangat penting dalam analitik IoT.

## Penutup

Data kotor (timestamp invalid, duplikat, nilai hilang, gap interval) berhasil dibersihkan
menjadi deret 10 menit yang konsisten tanpa nilai hilang. Dataset bersih ini menjadi dasar
deteksi anomali statistik pada Hands-on 8.